
# 39 多因子合成：IC、风险平价与机器学习

> **本章要解决的问题**：三个单因子的 IC 都是 0.05，合成之后是不是 0.15？等权、IC 加权、ICIR 加权、风险平价、ridge、stacking，到底差在哪？为什么合成后的换手能吃掉一半收益？

## 本章导读

单因子是原料，组合才是产品。第03章做了单因子报告，第04章讲过多因子的重复信息，第13章讲有限预算下的多因子组合，第34章讲组合优化。这一章把“多个因子怎么变成一个分数”讲完整：

1. 合成前处理：方向、去极值/标准化、正交化；
2. 权重方案：等权、IC 加权、ICIR 加权；
3. 风险视角：因子收益协方差、收缩估计、风险平价；
4. 学习视角：ridge、stacking、按时间切分；
5. 现实约束：换手、交易成本、权重平滑；
6. 对照评估：同一口径下比较 IC、ICIR、t 值、换手与净收益。

记住一句话：**合成不是“把 IC 加起来”，而是“在相关性、风险和成本之间做取舍”**。

### 学习目标

- 给定若干单因子面板与下一期收益，能统一方向、做截面去极值与标准化，并说明每一步改变了什么。
- 给定一组高度相关的因子，能用正交化（回归取残差）判断“增量信息还剩多少”。
- 给定单因子 IC 序列，能实现等权、IC 加权、ICIR 加权，并比较三者的权重与风险贡献。
- 给定因子多空收益的协方差，能用收缩估计与风险平价求权重，并验证各因子的风险贡献是否相等。
- 给定因子暴露与标签，能按时间切分训练/测试，用 ridge 与简单 stacking 做合成，并报告样本外 IC，避免未来信息泄露。
- 给定权重序列，能算换手与交易成本，把毛收益、净收益和平滑方案放在同一张表里对照。

### 先修

第04章的线性代数（共线、正交、矩阵）；第11章的机器学习与向前滚动验证；第13章的多因子组合；第20章的多重检验；会 numpy 的矩阵运算与随机数生成。

## 在量化研究中的位置

- **谁用**：因子研究员、组合经理、量化开发、风控。
- **输入**：单因子面板、下一期收益、股票池、协方差估计窗口、成本假设。
- **输出**：合成分数或权重、样本外 IC/ICIR、换手与净收益、方法对照表。
- **在流程的哪一步**：单因子初筛之后、组合优化与回测之前；它决定“多个因子怎么变成一个信号”。
- **失败模式**：方向搞反、用全样本 IC 定权重（前视）、共线因子重复加权、忽略换手成本、训练/测试泄漏。
- **最小工作流**：预处理 → 正交化 → 估权重 → 合成 → 换手/成本 → 样本外对照。

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="39 多因子合成：IC、风险平价与机器学习" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>39 多因子合成：IC、风险平价与机器学习｜知识图</title>
<desc>本章主线：方向、标准化、正交化先做好，再谈权重；合成之后必须做同日增量对照。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a39" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao39" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag39" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">39</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">多因子合成：IC、风险平价与机器学习</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：方向、标准化、正交化先做好，再谈权重；合成之后必须做同日增量对照。</text>
<rect x="43" y="96" width="130" height="76" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="108" y="126.6" font-size="12" fill="#1B3A5C" text-anchor="middle" font-weight="bold">方向统一</text>
<text x="108" y="145.1" font-size="9" fill="#6E7B8B" text-anchor="middle">+1 / −1</text>
<rect x="189" y="96" width="130" height="76" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="254" y="126.6" font-size="12" fill="#1B3A5C" text-anchor="middle" font-weight="bold">标准化</text>
<text x="254" y="145.1" font-size="9" fill="#6E7B8B" text-anchor="middle">z-score / rank</text>
<rect x="335" y="96" width="130" height="76" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="400" y="126.6" font-size="12" fill="#1B3A5C" text-anchor="middle" font-weight="bold">正交化</text>
<text x="400" y="145.1" font-size="9" fill="#6E7B8B" text-anchor="middle">残差去共线</text>
<rect x="481" y="96" width="130" height="76" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="546" y="119.8" font-size="12" fill="#1B3A5C" text-anchor="middle" font-weight="bold">定权</text>
<text x="546" y="138.4" font-size="9" fill="#6E7B8B" text-anchor="middle">等权 / IC / 风险平价 /</text>
<text x="546" y="151.9" font-size="9" fill="#6E7B8B" text-anchor="middle">ML</text>
<rect x="627" y="96" width="130" height="76" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="692" y="126.6" font-size="12" fill="#1B3A5C" text-anchor="middle" font-weight="bold">合成分数</text>
<text x="692" y="145.1" font-size="9" fill="#6E7B8B" text-anchor="middle">同日增量对照</text>
<path d="M 173 134 L 189 134" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a39)"/>
<path d="M 319 134 L 335 134" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a39)"/>
<path d="M 465 134 L 481 134" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a39)"/>
<path d="M 611 134 L 627 134" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a39)"/>
<rect x="45" y="198" width="320" height="178" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.5" rx="14"/>
<text x="60" y="218" font-size="10.5" fill="#1B3A5C" font-weight="bold">得分矩阵（标准化后）</text>
<text x="60" y="250" font-size="8.6" fill="#6E7B8B">动量</text>
<text x="60" y="274" font-size="8.6" fill="#6E7B8B">拥挤</text>
<text x="60" y="298" font-size="8.6" fill="#6E7B8B">库存</text>
<text x="60" y="322" font-size="8.6" fill="#6E7B8B">质量</text>
<rect x="100" y="234" width="46" height="22" fill="#1B3A5C" rx="3"/>
<text x="123" y="249" font-size="7.6" fill="#FFFFFF" text-anchor="middle">+0.8</text>
<rect x="148" y="234" width="46" height="22" fill="#E07B39" rx="3"/>
<text x="171" y="249" font-size="7.6" fill="#FFFFFF" text-anchor="middle">-1.2</text>
<rect x="196" y="234" width="46" height="22" fill="#E6EFF7" rx="3"/>
<text x="219" y="249" font-size="7.6" fill="#1B3A5C" text-anchor="middle">+0.3</text>
<rect x="244" y="234" width="46" height="22" fill="#1B3A5C" rx="3"/>
<text x="267" y="249" font-size="7.6" fill="#FFFFFF" text-anchor="middle">+1.4</text>
<rect x="292" y="234" width="46" height="22" fill="#E07B39" rx="3"/>
<text x="315" y="249" font-size="7.6" fill="#FFFFFF" text-anchor="middle">-0.5</text>
<rect x="100" y="258" width="46" height="22" fill="#E07B39" rx="3"/>
<text x="123" y="273" font-size="7.6" fill="#FFFFFF" text-anchor="middle">-0.6</text>
<rect x="148" y="258" width="46" height="22" fill="#1B3A5C" rx="3"/>
<text x="171" y="273" font-size="7.6" fill="#FFFFFF" text-anchor="middle">+1.1</text>
<rect x="196" y="258" width="46" height="22" fill="#E6EFF7" rx="3"/>
<text x="219" y="273" font-size="7.6" fill="#1B3A5C" text-anchor="middle">+0.2</text>
<rect x="244" y="258" width="46" height="22" fill="#E07B39" rx="3"/>
<text x="267" y="273" font-size="7.6" fill="#FFFFFF" text-anchor="middle">-0.9</text>
<rect x="292" y="258" width="46" height="22" fill="#E6EFF7" rx="3"/>
<text x="315" y="273" font-size="7.6" fill="#1B3A5C" text-anchor="middle">+0.7</text>
<rect x="100" y="282" width="46" height="22" fill="#E6EFF7" rx="3"/>
<text x="123" y="297" font-size="7.6" fill="#1B3A5C" text-anchor="middle">+0.4</text>
<rect x="148" y="282" width="46" height="22" fill="#E6EFF7" rx="3"/>
<text x="171" y="297" font-size="7.6" fill="#1B3A5C" text-anchor="middle">+0.5</text>
<rect x="196" y="282" width="46" height="22" fill="#E07B39" rx="3"/>
<text x="219" y="297" font-size="7.6" fill="#FFFFFF" text-anchor="middle">-1.3</text>
<rect x="244" y="282" width="46" height="22" fill="#E6EFF7" rx="3"/>
<text x="267" y="297" font-size="7.6" fill="#1B3A5C" text-anchor="middle">+0.1</text>
<rect x="292" y="282" width="46" height="22" fill="#1B3A5C" rx="3"/>
<text x="315" y="297" font-size="7.6" fill="#FFFFFF" text-anchor="middle">+1.0</text>
<rect x="100" y="306" width="46" height="22" fill="#E07B39" rx="3"/>
<text x="123" y="321" font-size="7.6" fill="#FFFFFF" text-anchor="middle">-0.2</text>
<rect x="148" y="306" width="46" height="22" fill="#E07B39" rx="3"/>
<text x="171" y="321" font-size="7.6" fill="#FFFFFF" text-anchor="middle">-0.7</text>
<rect x="196" y="306" width="46" height="22" fill="#1B3A5C" rx="3"/>
<text x="219" y="321" font-size="7.6" fill="#FFFFFF" text-anchor="middle">+0.9</text>
<rect x="244" y="306" width="46" height="22" fill="#E07B39" rx="3"/>
<text x="267" y="321" font-size="7.6" fill="#FFFFFF" text-anchor="middle">-0.4</text>
<rect x="292" y="306" width="46" height="22" fill="#E6EFF7" rx="3"/>
<text x="315" y="321" font-size="7.6" fill="#1B3A5C" text-anchor="middle">+0.6</text>
<text x="60" y="360" font-size="8.4" fill="#6E7B8B">颜色：深 = 高分，橙 = 负分</text>
<rect x="390" y="198" width="365" height="178" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.5" rx="14"/>
<text x="405" y="218" font-size="10.5" fill="#1B3A5C" font-weight="bold">合成权重：等权 vs IC 加权</text>
<text x="405" y="256" font-size="9" fill="#6E7B8B">动量</text>
<rect x="455" y="244" width="200" height="16" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1" rx="4"/>
<rect x="455" y="244" width="200" height="16" fill="#E07B39" stroke="#1B3A5C" stroke-width="1.1" rx="4"/>
<text x="661" y="256" font-size="8.8" fill="#1B3A5C">0.35</text>
<text x="405" y="286" font-size="9" fill="#6E7B8B">拥挤</text>
<rect x="455" y="274" width="200" height="16" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1" rx="4"/>
<rect x="455" y="274" width="142.9" height="16" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.1" rx="4"/>
<text x="603.9" y="286" font-size="8.8" fill="#1B3A5C">0.25</text>
<text x="405" y="316" font-size="9" fill="#6E7B8B">库存</text>
<rect x="455" y="304" width="200" height="16" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1" rx="4"/>
<rect x="455" y="304" width="125.7" height="16" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.1" rx="4"/>
<text x="586.7" y="316" font-size="8.8" fill="#1B3A5C">0.22</text>
<text x="405" y="346" font-size="9" fill="#6E7B8B">质量</text>
<rect x="455" y="334" width="200" height="16" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1" rx="4"/>
<rect x="455" y="334" width="102.9" height="16" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.1" rx="4"/>
<text x="563.9" y="346" font-size="8.8" fill="#1B3A5C">0.18</text>
<text x="405" y="364" font-size="8.4" fill="#6E7B8B">同一矩阵，不同加权 → 不同组合；必须同日对照</text>
<path d="M 400 174 Q 355 188 300 197" fill="none" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="4 3" marker-end="url(#ag39)"/>
<path d="M 546 174 Q 585 188 620 197" fill="none" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="4 3" marker-end="url(#ag39)"/>
</svg>

> **图**：本章主线：方向、标准化、正交化先做好，再谈权重；合成之后必须做同日增量对照。


In [1]:

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("numpy", np.__version__, "| matplotlib 中文字体已设置")


numpy 2.4.4 | matplotlib 中文字体已设置



## 1. 合成前处理：方向、标准化、正交化

**场景**：三个研究员各交来一个因子。A 的“价值”越大越好，B 的“动量”是反向的（IC 为负），C 的“质量”和“价值”截面相关系数 0.74。如果直接等权相加，会发生两件事：反向因子在拖后腿，两个相关因子在重复投票。

### 1.1 方向：先让符号有意义

方向的来源有两种：

1. **经济学方向**：价值（便宜的好）、质量（赚钱的好）、低波（稳的好）——符号由定义决定；
2. **统计方向**：用样本内 IC 的符号 `sign(mean(IC))` 决定。

本章用第二种做演示，但真实项目里要先看第一种：如果统计方向和经济逻辑相反，大概率是数据或口径出了问题，不该简单地“乘个 -1”了事。方向统一之后，所有因子的 IC 都应该是正的，后面的加权才有意义。

### 1.2 标准化：去极值 + 截面 zscore

不同因子的量纲差几个数量级：动量是收益率，换手率是百分比，财务比率可能是倍数。加权之前必须把它们放到同一把尺子上。标准流程是：

1. **去极值**：把超过 3 倍标准差的取值截断（winsorize），防止一两个异常值主导权重；
2. **截面 zscore**：对每个交易日横截面做 $(x-\mu_t)/\sigma_t$，消除量纲和水平差异。

注意是**截面**标准化，不是全样本标准化。用全样本均值和方差会把未来信息带进历史（第08章的未来函数），而且会让不同时期的分数不可比。

### 1.3 正交化：残差才是增量信息

“质量”和“价值”相关 0.74，说明它们讲的是同一件事的大部分。正交化的做法是：把“质量”对“价值”做截面回归，取残差作为新的“质量残差”：

$$\tilde q_t = q_t - \big(a_t + b_t v_t\big).$$

**公式拆解**
- **符号**：$q_t$ 是“质量”在第 $t$ 期的截面值，$v_t$ 是“价值”截面，$a_t,b_t$ 是每期把 $q_t$ 对 $v_t$ 回归的截距与斜率，$\tilde q_t$ 是残差。
- **结构**：从 $q_t$ 减去能被 $v_t$ 解释的部分 $a_t+b_tv_t$，残差与价值正交，代表“价值解释不了的那部分质量”，也就是增量信息。
- **代入**：本章两者截面相关 0.743，质量 IC 从 0.081 降到 0.028；若某期拟合值 0.87、$q_t=1.2$，残差就是 0.33。
- **边界**：$v_t$ 截面方差为 0 时回归无解要跳过该期，基准因子本身有噪声，正交化也可能损失有用的共同信息。
- **用途**：判断新因子是否带来增量、去掉重复投票，是入库前的标准检查；残差 IC 小不等于因子无效，只说明独立增量少。

残差与“价值”正交，代表“价值解释不了的那部分质量”。如果残差的 IC 从 0.081 掉到 0.028，说明原因子的大部分预测力来自和价值共享的信息，**增量价值有限**；如果残差 IC 基本不掉，说明它确实带来了新信息。

### 1.4 代码：预处理与单因子 IC

下面构造三个月度面板（120 个月 × 300 只股票），其中“质量”与“价值”刻意相关，“动量”刻意反向，然后走一遍方向统一、去极值、zscore、正交化和 IC 计算。



In [2]:

rng = np.random.default_rng(3901)
T, N = 120, 300


def zscore(a):
    mu = a.mean(axis=1, keepdims=True)
    sd = a.std(axis=1, keepdims=True) + 1e-12
    return (a - mu) / sd


def ic_series(signal, label):
    return np.array([np.corrcoef(signal[t], label[t])[0, 1] for t in range(signal.shape[0])])


common = rng.normal(size=(T, N))
f_val = zscore(common + 0.7 * rng.normal(size=(T, N)))
f_qual = zscore(0.85 * common + 0.4 * rng.normal(size=(T, N)))
f_mom = zscore(-0.45 * common + rng.normal(size=(T, N)))

# 月度收益（小数）：信号系数 ~0.003、噪声 0.055，单因子 IC 落在 0.04–0.09
y = 0.0030 * f_val - 0.0022 * f_mom + 0.0010 * f_qual + rng.normal(0, 0.055, size=(T, N))

raw = {"价值": f_val, "质量": f_qual, "动量": f_mom}
raw_ic = {}
for name, sig in raw.items():
    ic = ic_series(zscore(np.clip(sig, -3, 3)), y)
    raw_ic[name] = ic
    print(f"{name}: IC 均值 {ic.mean(): .3f}  ICIR {ic.mean() / ic.std(ddof=1): .2f}"
          f"  t 值 {ic.mean() / (ic.std(ddof=1) / np.sqrt(len(ic))): .2f}")

direction = {name: int(np.sign(raw_ic[name].mean())) for name in raw}
print("方向", direction)

aligned = {name: direction[name] * zscore(np.clip(zscore(raw[name]), -3, 3)) for name in raw}


def orthogonalize(target, base):
    out = np.zeros_like(target)
    for t in range(target.shape[0]):
        design = np.column_stack([np.ones(target.shape[1]), base[t]])
        beta, *_ = np.linalg.lstsq(design, target[t], rcond=None)
        out[t] = target[t] - design @ beta
    return zscore(out)


qual_resid = orthogonalize(aligned["质量"], aligned["价值"])
corr_vq = float(np.mean([np.corrcoef(aligned["价值"][t], aligned["质量"][t])[0, 1] for t in range(T)]))
ic_q = ic_series(aligned["质量"], y).mean()
ic_q_resid = ic_series(qual_resid, y).mean()
print("价值-质量 平均截面相关", round(corr_vq, 3))
print(f"质量 IC {ic_q:.3f} -> 正交化后 {ic_q_resid:.3f}")


价值: IC 均值  0.084  ICIR  1.57  t 值  17.19
质量: IC 均值  0.081  ICIR  1.51  t 值  16.58
动量: IC 均值 -0.079  ICIR -1.41  t 值 -15.48
方向 {'价值': 1, '质量': 1, '动量': -1}
价值-质量 平均截面相关 0.743
质量 IC 0.081 -> 正交化后 0.028



**读数**：价值、质量、动量的 IC 分别是 0.084、0.081、−0.079，ICIR 分别是 1.57、1.51、−1.41，t 值都在 15 以上。方向表把动量翻成 −1，统一后三个因子都是“越大越好”。价值与质量的平均截面相关 0.743；质量 IC 从 0.081 掉到 0.028，说明它约三分之二的预测力来自和价值共享的信息。

**常见错误**：

1. 用全样本均值/方差标准化，把未来信息带进历史。
2. 方向只靠代码里乘 −1，因子卡里不记录，合成时有人加有人减。
3. 看到相关性高就直接删因子，而不是先看残差还剩多少信息。

---



## 2. 权重方案：等权、IC、ICIR

合成公式都是一样的：$s_t = \sum_k w_k z_{k,t}$，区别只在权重 $w_k$ 怎么来。

### 2.1 等权

最朴素的方案：$w_k = 1/K$。优点是对估计误差不敏感、换手低；缺点是默认“所有因子一样好”，而现实中 IC 可能差三倍。等权是最重要的基准——任何聪明方法都必须先打赢等权。

### 2.2 IC 加权

用每个因子的平均 IC 当权重：$w_k \propto \max(\overline{IC}_k, 0)$，再归一化。直观含义是“预测力强的因子多给钱”。工程上有两点要注意：负 IC 要截断（否则反向因子拿负权重，等于又做一次方向判断）；IC 必须用滚动窗口估计，不能用全样本。

**公式拆解**
- **符号**：$\hat\Sigma$ 是收缩后的协方差矩阵，$S$ 是样本协方差，$F$ 是结构化目标（本章取对角阵），$\lambda\in[0,1]$ 是收缩强度。
- **结构**：$\lambda F+(1-\lambda)S$ 是凸组合，对角元保持不变、非对角元按 $1-\lambda$ 向 0 拉近，矩阵条件数下降、求逆更稳，代价是引入偏差。
- **代入**：$S_{12}=0.293\times10^{-4}$、$\lambda=0.3$ 时收缩后为 $0.7\times0.293\approx0.205$，相关系数从 0.301 降到约 0.21。
- **边界**：$\lambda=0$ 用纯样本协方差（小样本可能不可逆），$\lambda=1$ 等于假设完全不相关且方差仍可能估不准；样本越短、因子越多，$\lambda$ 应越大。
- **用途**：给组合优化与风险平价提供稳定的协方差输入；它不是相关性预测，结构性突变要另做压力情景。

### 2.3 ICIR 加权

IC 的均值除以 IC 的标准差：$w_k \propto \max(ICIR_k, 0)$。它等价于对“稳定性”加权：两个因子平均 IC 相同，波动小的那个拿更多权重。第02章讲过 $ICIR = t/\sqrt{T}$，所以 ICIR 加权也可以理解为“按统计显著性加权”。

### 2.4 三种方案的风险在哪

| 方案 | 权重公式 | 优点 | 风险 |
|---|---|---|---|
| 等权 | $1/K$ | 稳、换手低 | 默认一样好 |
| IC 加权 | $\propto \overline{IC}$ | 简单、可解释 | 权重集中在少数因子 |
| ICIR 加权 | $\propto \overline{IC}/\sigma_{IC}$ | 偏好稳定因子 | 对样本长度敏感 |

三种方案都只看“预测力”，没有看“因子之间是否同步”。这正是下一节要补的。

---

## 3. 风险视角：协方差收缩与风险平价

### 3.1 为什么看因子收益的协方差

权重不该只看 IC，还要看因子之间的“同步性”。如果两个因子高度同步，各给 30% 权重，组合实际上只承担了一种风险。这里用**因子多空组合的月度收益**估协方差：对每个因子，每月做多信号最强 20%、做空最弱 20%，得到一条收益序列 $r^f_t$。

### 3.2 收缩估计

样本协方差在小样本、多因子时会高估极端相关性、低估方差。常用做法是向一个结构化目标收缩：

$$\hat\Sigma = \lambda F + (1-\lambda) S,$$

**公式拆解**
- **符号**：$w_k$ 是第 $k$ 个因子的权重（无单位、和为 1），$\overline{IC}_k$ 是该因子平均截面 IC（相关系数），$z_{k,t}$ 是第 $t$ 期截面标准化分数。
- **结构**：合成分数是 $s_t=\sum_kw_kz_{k,t}$，权重 $w_k=\max(\overline{IC}_k,0)/\sum_j\max(\overline{IC}_j,0)$，平均 IC 越高权重越大、负 IC 被截断防止反向因子拿负权重。
- **代入**：三因子 IC 为 $(0.0836,0.0278,0.0793)$ 时权重为 $(0.438,0.146,0.416)$；若某期分数为 $(1,0.5,-0.5)$，合成分数约 0.438 加 0.073 减 0.208 等于 0.303。
- **边界**：所有因子 IC 为负或为 0 时分母为 0、公式未定义，要回退等权；用全样本 IC 是未来函数，样本越短权重越不稳。
- **用途**：最常用、最容易解释的合成权重，适合当作基准；它不控制风险，权重可能集中在两三个因子上，风险贡献严重不均。

其中 $S$ 是样本协方差，$F$ 取对角阵，$\lambda$ 控制收缩强度。样本越短、因子越多，$\lambda$ 应该越大。本章取 $\lambda=0.3$，即把非对角元素向 0 拉 30%。

### 3.3 风险平价：等风险贡献

风险贡献定义为 $RC_i = w_i(\Sigma w)_i / \sigma_p$，$\sigma_p=\sqrt{w^\top\Sigma w}$。风险平价要求 $w_i(\Sigma w)_i = w_j(\Sigma w)_j$ 对所有 $i,j$ 成立。最常用的算法是迭代：从等权出发，反复用 $w \leftarrow 1/(\Sigma w)$ 更新并归一化，直到稳定。它不是“波动率倒数加权”的严格等价（只有因子两两不相关时才等价），但实践中足够好。

### 3.4 代码：四种权重一次算完

下面这段代码先算因子多空收益和协方差，再依次得到等权、IC、ICIR、风险平价四套权重，最后打印每套权重的风险贡献。末尾再加一个“压力协方差”，用来看相关性上升、动量波动翻倍时权重会怎么变。



In [3]:

def long_short_return(signal, q=0.2):
    k = int(N * q)
    rets = np.zeros(T)
    for t in range(T):
        order = np.argsort(-signal[t])
        rets[t] = y[t, order[:k]].mean() - y[t, order[-k:]].mean()
    return rets


factors = {"价值": aligned["价值"], "质量": qual_resid, "动量": aligned["动量"]}
FR = np.column_stack([long_short_return(sig) for sig in factors.values()])
print("因子多空月收益 均值", np.round(FR.mean(axis=0) * 100, 3), "% | 年化波动",
      np.round(FR.std(axis=0, ddof=1) * np.sqrt(12) * 100, 2), "%")

ics = np.array([ic_series(sig, y).mean() for sig in factors.values()])
icirs = np.array([ic_series(sig, y).mean() / ic_series(sig, y).std(ddof=1) for sig in factors.values()])
print("IC", np.round(ics, 4), "| ICIR", np.round(icirs, 3))

pos_ic = np.clip(ics, 0, None)
pos_icir = np.clip(icirs, 0, None)
w_eq = np.ones(3) / 3
w_ic = pos_ic / pos_ic.sum()
w_icir = pos_icir / pos_icir.sum()

S = np.cov(FR.T, ddof=1)
S_shrunk = 0.3 * np.diag(np.diag(S)) + 0.7 * S
print("样本相关矩阵\n", np.round(np.corrcoef(FR.T), 3))
print("样本协方差（×1e4）\n", np.round(S * 1e4, 3))
print("收缩后（×1e4）\n", np.round(S_shrunk * 1e4, 3))


def risk_parity(cov, iters=1000):
    n = cov.shape[0]
    w = np.ones(n) / n
    for _ in range(iters):
        w = np.maximum(1.0 / (cov @ w), 1e-10)
        w = w / w.sum()
    return w


def risk_contrib(w, cov):
    var = float(w @ cov @ w)
    return w * (cov @ w) / var


w_rp = risk_parity(S_shrunk)

vol = np.sqrt(np.diag(S_shrunk))
corr_stress = np.full((3, 3), 0.8)
vol_stress = vol * np.array([1.0, 1.0, 2.0])
S_stress = np.outer(vol_stress, vol_stress) * corr_stress
np.fill_diagonal(S_stress, vol_stress ** 2)
w_rp_stress = risk_parity(S_stress)
print("压力协方差下风险平价权重", np.round(w_rp_stress, 3),
      "| 风险贡献", np.round(risk_contrib(w_rp_stress, S_stress), 3))
print(f"{'方案':<8}{'w1':>8}{'w2':>8}{'w3':>8}{'风险贡献':>24}{'年化波动':>10}")
for name, w in [("等权", w_eq), ("IC", w_ic), ("ICIR", w_icir), ("风险平价", w_rp),
                ("压力平价", w_rp_stress)]:
    vol_p = float(np.sqrt(w @ S_shrunk @ w)) * np.sqrt(12)
    rc = np.round(risk_contrib(w, S_shrunk), 3)
    print(f"{name:<8}{w[0]:>8.3f}{w[1]:>8.3f}{w[2]:>8.3f}{str(rc):>24}{vol_p:>10.2%}")


因子多空月收益 均值 [1.212 0.417 1.259] % | 年化波动 [3.18 3.34 3.68] %
IC [0.0836 0.0278 0.0793] | ICIR [1.569 0.519 1.413]
样本相关矩阵
 [[1.    0.031 0.301]
 [0.031 1.    0.055]
 [0.301 0.055 1.   ]]
样本协方差（×1e4）
 [[0.84  0.027 0.293]
 [0.027 0.93  0.056]
 [0.293 0.056 1.127]]
收缩后（×1e4）
 [[0.84  0.019 0.205]
 [0.019 0.93  0.039]
 [0.205 0.039 1.127]]
压力协方差下风险平价权重 [0.42  0.399 0.181] | 风险贡献 [0.333 0.333 0.333]
方案            w1      w2      w3                    风险贡献      年化波动
等权         0.333   0.333   0.333     [0.311 0.289 0.401]     2.14%
IC         0.438   0.146   0.416     [0.437 0.051 0.512]     2.34%
ICIR       0.448   0.148   0.404     [0.456 0.053 0.491]     2.34%
风险平价       0.347   0.356   0.297     [0.333 0.333 0.333]     2.12%
压力平价       0.420   0.399   0.181     [0.443 0.409 0.147]     2.13%



**读数**：三个多空组合的月均收益是 1.212%、0.417%、1.259%，年化波动 3.18%、3.34%、3.68%。价值与动量的相关系数 0.301，压力情景下会更高。IC 加权把权重压给价值和动量（0.438/0.146/0.416），风险贡献变成 0.437/0.051/0.512，**质量因子的风险贡献只剩 5%**——这是“按预测力加权”的典型后果。风险平价给出 0.347/0.356/0.297，风险贡献三项各 1/3；压力情景下权重变成 0.420/0.399/0.181，自动把高波动的动量因子降下来。

**常见错误**：

1. 用全样本 IC 估权重——权重里已经含了未来信息。
2. 把风险平价当成“波动率倒数加权”——只有在因子两两不相关时才成立。
3. 只报权重不报风险贡献——权重 30% 不等于风险 30%。

---



## 4. 机器学习合成：ridge 与 stacking

### 4.1 为什么是 ridge 而不是 OLS

把 K 个因子当特征、下一期收益当标签，做一个线性模型，权重就是回归系数。普通最小二乘在因子共线时系数会爆炸（第04章讲过），而 **ridge** 在目标里加了 $\alpha\lVert\beta\rVert^2$ 的惩罚，把系数往 0 拉，换来更稳的样本外表现。$\alpha$ 越大，收缩越强。

### 4.2 时间切分：先切时间，再切样本

因子研究的样本是“时间 × 截面”的面板。切分必须按**时间**：前 84 个月训练、后 36 个月测试（本章共 120 个月）。绝不能随机打乱，否则同一个月的数据同时出现在训练集和测试集，IC 会好得离谱。

### 4.3 stacking：用模型学“怎么组合模型”

stacking 分两层：

1. 第一层：三个单因子 ridge 加一个全因子 ridge，各自给出预测；
2. 生成**折外（OOF）预测**：把训练期切成 5 段，轮流留一段做验证、其余训练，拼出一条“没在自己训练数据上做过预测”的序列；
3. 第二层：拿 OOF 预测当特征、真实收益当标签，再训练一个简单线性模型。

关键是 OOF：如果直接拿第一层在训练集上的预测训练第二层，第二层会学到“第一层过拟合的部分”，样本外必然失望。

### 4.4 代码：ridge 与 stacking



In [4]:

X = np.stack([aligned["价值"], qual_resid, aligned["动量"]], axis=2)
Xp = X.reshape(T * N, 3)
yp = y.reshape(T * N)
cut = int(T * 0.7)
print("训练期", cut, "个月，测试期", T - cut, "个月，面板行数", Xp.shape[0])


def fit_ridge(X, y, alpha):
    x_mean, y_mean = X.mean(axis=0), y.mean()
    Xc = X - x_mean
    beta = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(X.shape[1]), Xc.T @ (y - y_mean))
    return beta, x_mean, y_mean


def predict_ridge(model, X):
    beta, x_mean, y_mean = model
    return (X - x_mean) @ beta + y_mean


def panel_ic(pred, label, periods):
    pred2 = pred.reshape(periods, N)
    label2 = label.reshape(periods, N)
    return np.array([np.corrcoef(pred2[t], label2[t])[0, 1] for t in range(periods)])


train = slice(0, cut * N)
test = slice(cut * N, T * N)
Xtr, ytr = Xp[train], yp[train]
Xte, yte = Xp[test], yp[test]

ridge = fit_ridge(Xtr, ytr, alpha=5.0)
pred_ridge = predict_ridge(ridge, Xte)
ic_ridge = panel_ic(pred_ridge, yte, T - cut)
print("ridge 系数", np.round(ridge[0], 4), "| 截距", round(float(ridge[2]), 5))
print("ridge 测试 IC", round(ic_ridge.mean(), 4), "ICIR", round(ic_ridge.mean() / ic_ridge.std(ddof=1), 2))


def oof_predict(X, y, alpha, folds=5):
    periods = X.shape[0] // N
    X3 = X.reshape(periods, N, -1)
    y2 = y.reshape(periods, N)
    oof = np.zeros_like(y2)
    edges = np.linspace(0, periods, folds + 1).astype(int)
    for k in range(folds):
        a, b = edges[k], edges[k + 1]
        mask = np.ones(periods, bool)
        mask[a:b] = False
        Xa = X3[mask].reshape(-1, X3.shape[2])
        ya = y2[mask].ravel()
        model = fit_ridge(Xa, ya, alpha)
        oof[a:b] = predict_ridge(model, X3[a:b].reshape(-1, X3.shape[2])).reshape(b - a, N)
    return oof


oof_cols, test_cols = [], []
for j in range(3):
    oof_cols.append(oof_predict(Xtr[:, j:j + 1], ytr, alpha=1.0).ravel())
    model_j = fit_ridge(Xtr[:, j:j + 1], ytr, alpha=1.0)
    test_cols.append(predict_ridge(model_j, Xte[:, j:j + 1]))
oof_full = oof_predict(Xtr, ytr, alpha=5.0)
test_cols.append(pred_ridge)

Z_tr = np.column_stack(oof_cols + [oof_full.ravel()])
Z_te = np.column_stack(test_cols)
stack = fit_ridge(Z_tr, ytr, alpha=1.0)
pred_stack = predict_ridge(stack, Z_te)
ic_stack = panel_ic(pred_stack, yte, T - cut)
print("stacking 二层系数", np.round(stack[0], 4))
print("stacking 测试 IC", round(ic_stack.mean(), 4), "ICIR", round(ic_stack.mean() / ic_stack.std(ddof=1), 2))

pred_eq = Xte.mean(axis=1)
ic_eq = panel_ic(pred_eq, yte, T - cut)
print("等权合成分 测试 IC", round(ic_eq.mean(), 4), "ICIR", round(ic_eq.mean() / ic_eq.std(ddof=1), 2))


训练期 84 个月，测试期 36 个月，面板行数 36000
ridge 系数 [0.0038 0.0011 0.0027] | 截距 1e-05
ridge 测试 IC 0.1006 ICIR 1.96
stacking 二层系数 [0.2207 0.032  0.1771 0.3161]
stacking 测试 IC 0.1008 ICIR 1.91
等权合成分 测试 IC 0.0941 ICIR 2.07



**读数**：ridge 系数是 0.0038/0.0011/0.0027，方向与真实结构一致（价值、动量的贡献更大）。样本外测试段：ridge IC 0.1006、ICIR 1.96；stacking IC 0.1008、ICIR 1.91；等权合成分 IC 0.0941、ICIR 2.07。**stacking 并没有赢过简单等权**——这是常见结果：因子只有三个、相关性又不强时，复杂模型的边际收益很小，却增加了复杂度和过拟合风险。

**常见错误**：

1. 随机切分面板——同月数据同时进训练和测试。
2. 第二层直接用第一层在训练集上的预测，不用 OOF。
3. 用同一段样本既调 $\alpha$ 又报测试 IC。

---

## 5. 换手、成本与权重平滑

### 5.1 换手：合成方案的真实代价

组合每期都要调仓。换手定义为

$$TO_t = \frac{1}{2}\sum_i |w_{i,t} - w_{i,t-1}|.$$

月换手 0.9 意味着几乎每月把整个组合换一遍。成本 ≈ 换手 × 单边成本，年化成本 ≈ 平均换手 × 单边成本 × 12。两个 IC 相同的方案，换手高一倍，净收益可能差一倍。

### 5.2 平滑：用 alpha 换成本

平滑的做法是让持仓慢慢靠近目标：

$$w_t = (1-\lambda)w_{t-1} + \lambda w^{target}_t.$$

等价于指数移动平均。半衰期越短，越贴近目标、换手越高；半衰期越长，越省成本、但信号衰减越多。平滑之后记得重新归一化，否则组合暴露会不知不觉变小。

### 5.3 代码：换手、成本与成本扫描



In [5]:

def weights_from_score(score, k):
    W = np.zeros_like(score)
    for t in range(score.shape[0]):
        order = np.argsort(-score[t])
        W[t, order[:k]] = 0.5 / k
        W[t, order[-k:]] = -0.5 / k
    return W


def turnover(W):
    dw = np.diff(np.vstack([np.zeros((1, W.shape[1])), W]), axis=0)
    return 0.5 * np.abs(dw).sum(axis=1)


score_test = X.mean(axis=2)[cut:]
k = int(N * 0.10)
W_raw = weights_from_score(score_test, k)
turn_raw = turnover(W_raw)
gross_raw = (W_raw * y[cut:]).sum(axis=1)
cost_bps = 10
net_raw = gross_raw - turn_raw * cost_bps / 1e4

W_sm = np.zeros_like(W_raw)
prev = np.zeros(N)
for t in range(W_raw.shape[0]):
    prev = 0.5 * prev + 0.5 * W_raw[t]
    W_sm[t] = prev
W_sm = W_sm / np.abs(W_sm).sum(axis=1, keepdims=True)
turn_sm = turnover(W_sm)
gross_sm = (W_sm * y[cut:]).sum(axis=1)
net_sm = gross_sm - turn_sm * cost_bps / 1e4

h = 3
lam = 1 - 0.5 ** (1 / h)
W_ew = np.zeros_like(W_raw)
prev = np.zeros(N)
for t in range(W_raw.shape[0]):
    prev = (1 - lam) * prev + lam * W_raw[t]
    W_ew[t] = prev
W_ew = W_ew / np.abs(W_ew).sum(axis=1, keepdims=True)
turn_ew = turnover(W_ew)
gross_ew = (W_ew * y[cut:]).sum(axis=1)
net_ew = gross_ew - turn_ew * cost_bps / 1e4

for label, turn, gross, net in [
    ("原始权重", turn_raw, gross_raw, net_raw),
    ("50% 平滑", turn_sm, gross_sm, net_sm),
    ("半衰期 3 月", turn_ew, gross_ew, net_ew),
]:
    print(f"{label:<10} 平均换手 {turn.mean():.3f} | 年化毛收益 {gross.mean() * 12:.2%}"
          f" | 年化净收益 {net.mean() * 12:.2%} | 年化成本 {turn.mean() * cost_bps / 1e4 * 12:.2%}")

print("成本扫描（年化净收益）")
print(f"{'单边成本':<10}{'原始':>10}{'50% 平滑':>12}{'半衰期 3 月':>14}")
for bp in (10, 50, 100, 150):
    row = []
    for turn, gross in [(turn_raw, gross_raw), (turn_sm, gross_sm), (turn_ew, gross_ew)]:
        row.append((gross - turn * bp / 1e4).mean() * 12)
    print(f"{bp:>4d} bp    {row[0]:>10.2%}{row[1]:>12.2%}{row[2]:>14.2%}")


原始权重       平均换手 0.895 | 年化毛收益 10.17% | 年化净收益 9.09% | 年化成本 1.07%
50% 平滑     平均换手 0.509 | 年化毛收益 6.25% | 年化净收益 5.64% | 年化成本 0.61%
半衰期 3 月    平均换手 0.286 | 年化毛收益 4.04% | 年化净收益 3.70% | 年化成本 0.34%
成本扫描（年化净收益）
单边成本              原始      50% 平滑       半衰期 3 月
  10 bp         9.09%       5.64%         3.70%
  50 bp         4.79%       3.20%         2.32%
 100 bp        -0.58%       0.14%         0.61%
 150 bp        -5.95%      -2.91%        -1.11%



**读数**：原始权重平均换手 0.895，年化毛收益 10.17%、净收益 9.09%，年化成本 1.07%；50% 平滑把换手降到 0.509，毛收益降到 6.25%，净收益 5.64%；半衰期 3 月的指数平滑换手只有 0.286，毛收益 4.04%、净收益 3.70%。**平滑省下的成本远小于 alpha 衰减的损失**。成本扫描显示，只有单边成本升到 100bp 以上，平滑才开始在净收益上占优（约 0.14% 与 −0.58%）。结论：平滑不是“越平滑越好”，要先算清楚 alpha 的半衰期和真实成本。

**常见错误**：

1. 只比毛收益不看换手，或只比换手不看净收益。
2. 平滑后忘记归一化，组合暴露越来越小。
3. 用 1bp 的“理想成本”做决策，实盘冲击成本是 10–50bp 量级。

---

## 6. 对照评估：同一口径下比较

### 6.1 评估维度

比较合成方法至少要看四组指标：

- **预测力**：IC 均值、ICIR、t 值；
- **稳定性**：IC 胜率、分组单调性、分年表现；
- **成本**：换手、年化成本、净收益；
- **样本外**：权重只能用训练段估，测试段评估。

最容易犯的错是“样本内比样本外”：用全样本 IC 定权重，再用同一样本报 IC，任何方法都会显得很神。

### 6.2 代码：六种方法的同一张表



In [6]:

ics_tr = np.array([ic_series(factors[name][:cut], y[:cut]).mean() for name in factors])
icirs_tr = np.array([ic_series(factors[name][:cut], y[:cut]).mean() /
                     ic_series(factors[name][:cut], y[:cut]).std(ddof=1) for name in factors])
print("训练段 IC", np.round(ics_tr, 4), "| 训练段 ICIR", np.round(icirs_tr, 3))

S_tr = np.cov(FR[:cut].T, ddof=1)
S_tr_shrunk = 0.3 * np.diag(np.diag(S_tr)) + 0.7 * S_tr
w_ic_tr = np.clip(ics_tr, 0, None) / np.clip(ics_tr, 0, None).sum()
w_icir_tr = np.clip(icirs_tr, 0, None) / np.clip(icirs_tr, 0, None).sum()
w_rp_tr = risk_parity(S_tr_shrunk)
print("训练段权重：IC", np.round(w_ic_tr, 3), "| ICIR", np.round(w_icir_tr, 3),
      "| 风险平价", np.round(w_rp_tr, 3))

rows = []
for name, w in [("等权", w_eq), ("IC 加权", w_ic_tr), ("ICIR 加权", w_icir_tr), ("风险平价", w_rp_tr)]:
    score = sum(w[i] * factors[n][cut:] for i, n in enumerate(factors))
    ic = ic_series(zscore(score), y[cut:])
    rows.append((name, ic.mean(), ic.mean() / ic.std(ddof=1),
                 ic.mean() / (ic.std(ddof=1) / np.sqrt(len(ic)))))

for name, pred in [("ridge", pred_ridge), ("stacking", pred_stack)]:
    ic = ic_series(pred.reshape(T - cut, N), y[cut:])
    rows.append((name, ic.mean(), ic.mean() / ic.std(ddof=1),
                 ic.mean() / (ic.std(ddof=1) / np.sqrt(len(ic)))))

print(f"{'方法':<10}{'IC':>9}{'ICIR':>9}{'t 值':>9}")
for name, ic, icir, tval in rows:
    print(f"{name:<10}{ic:>9.4f}{icir:>9.3f}{tval:>9.2f}")


训练段 IC [0.0856 0.0298 0.076 ] | 训练段 ICIR [1.676 0.56  1.33 ]
训练段权重：IC [0.447 0.156 0.397] | ICIR [0.47  0.157 0.373] | 风险平价 [0.365 0.352 0.283]
方法               IC     ICIR      t 值
等权           0.0941    2.070    12.42
IC 加权        0.1014    2.007    12.04
ICIR 加权      0.1010    1.989    11.94
风险平价         0.0925    2.022    12.13
ridge        0.1006    1.958    11.75
stacking     0.1008    1.907    11.44



**读数**：训练段 IC 是 0.0856、0.0298、0.0760，据此得到的 IC 权重是 0.447/0.156/0.397、ICIR 权重 0.470/0.157/0.373、风险平价 0.365/0.352/0.283。测试段六个方法的 IC 都在 0.09–0.10：等权 0.0941、IC 加权 0.1014、ICIR 加权 0.1010、风险平价 0.0925、ridge 0.1006、stacking 0.1008。**差距远小于“三个 0.05 相加约等于 0.15”的想象**。IC 加权和两个机器学习方法在 IC 上略高，但 ICIR/t 值没有明显优势，而且还要付出换手和复杂度的代价。因子少、相关性弱时，简单方案的性价比常常最高。

**常见错误**：

1. 在样本内比较方法，谁都能赢。
2. 只比 IC 不比换手，忽略“高 IC 但高成本”的方法。
3. 看到机器学习就默认更好，不设简单基准。

---

## 本章小结

### 一张表

| 环节 | 关键问题 | 手段 |
|---|---|---|
| 方向 | IC 的符号对不对 | 经济学方向优先，统计方向兜底 |
| 标准化 | 量纲不可比 | 去极值 3σ + 截面 zscore |
| 正交化 | 因子重复投票 | 对已有因子回归取残差 |
| 等权 | 最稳的基准 | $1/K$ |
| IC/ICIR 加权 | 强者多配 | $\propto \overline{IC}$ 或 $\propto ICIR$ |
| 风险平价 | 风险分散 | 等风险贡献，迭代求解 |
| 机器学习 | 非线性/自动加权 | ridge + OOF stacking，按时间切分 |
| 成本 | 合成后的真实收益 | 换手 × 成本，再做平滑权衡 |

### 八句话

1. 合成前先统一方向，否则加权没有意义。
2. 截面标准化，不能用全样本参数。
3. 相关因子先正交化，再谈增量信息。
4. 等权是最重要的基准。
5. IC 加权的风险会集中在少数因子。
6. 风险平价看的是风险贡献，不是权重。
7. 机器学习要按时间切分，并用 OOF 做 stacking。
8. 合成后的换手与成本，可能比方法本身更重要。

---

## 本章习题

**1.** 方向统一、去极值和截面 zscore 各解决什么问题？

**2.** 一个因子与已有因子相关 0.74，正交化后残差 IC 从 0.081 掉到 0.028，说明了什么？

**3.** IC 加权和 ICIR 加权的主要区别是什么？

**4.** 风险贡献怎么定义？为什么权重 30% 不等于风险 30%？

**5.** 为什么风险平价常用“迭代 $w \leftarrow 1/(\Sigma w)$”求解？

**6.** ridge 相比 OLS 为什么更适合多因子权重估计？

**7.** stacking 中为什么要用 OOF 预测？随机切分面板有什么问题？

**8.** 平滑能把换手降下来，但为什么净收益可能更差？什么情况下平滑才划算？

<details><summary>参考答案</summary>

**1.** 方向解决“符号是否一致”，去极值解决“异常值主导权重”，截面 zscore 解决“量纲与水平不可比”。

**2.** 它约三分之二的预测力来自与已有因子共享的信息，独立增量有限；如果要保留它，应使用残差版本。

**3.** IC 加权按预测力大小配权重，ICIR 加权还要除以 IC 的波动，偏好更稳定的因子。

**4.** $RC_i = w_i(\Sigma w)_i/\sigma_p$，同时受权重、波动和相关性影响；权重相同的两个因子，波动大、相关性高的那个风险贡献更大。

**5.** 该迭代的不动点满足 $w_i(\Sigma w)_i$ 为常数，正是等风险贡献的条件；实现简单、无需通用优化器。

**6.** 因子共线时 OLS 系数方差大、容易爆炸；ridge 的 $\ell_2$ 惩罚把系数向 0 收缩，偏差换方差，样本外更稳。

**7.** OOF 保证第二层的训练特征不是第一层在“自己见过的数据”上给出的预测，避免学到过拟合部分；随机切分面板会让同月数据同时出现在训练和测试集，造成泄漏。

**8.** 平滑省下的是成本，损失的是 alpha 的时效；当 alpha 衰减快、成本低时净收益更差，只有成本足够高（本例 100bp 以上）或 alpha 更持久时平滑才划算。

</details>

---

## 参考资料

- Grinold, R. & Kahn, R. *Active Portfolio Management*（IC、ICIR 与组合构建）。
- Maillard, S., Roncalli, T. & Teiletche, J. "The Properties of Equally Weighted Risk Contribution Portfolios", *Journal of Portfolio Management*, 2010。
- Ledoit, O. & Wolf, M. "A Well-Conditioned Estimator for Large-Dimensional Covariance Matrices", *Journal of Multivariate Analysis*, 2004。
- Hastie, T., Tibshirani, R. & Friedman, J. *The Elements of Statistical Learning*（ridge 与 stacking）。
- scikit-learn 文档：<https://scikit-learn.org>
- 第02、04、11、13、20、37章。
